# 01 — Data Exploration: Allen Human Brain Atlas

This notebook explores the Allen Human Brain Atlas microarray and RNA-seq datasets,
with a focus on mechanosensitive ion channel genes implicated in focused ultrasound
(FUS) neuromodulation.

**Reference:** Bhatt et al. 2024, *PMC10423872*

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src import gene_lists
from src import data_loader

sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

## 1. Gene families of interest

List all mechanosensitive ion channel genes organized by family.

In [ ]:
gene_df = pd.DataFrame([
    {'gene_symbol': g, 'alias': gene_lists.ALL_MECHANOSENSITIVE[g], 'family': gene_lists.get_family(g)}
    for g in gene_lists.ALL_MECHANOSENSITIVE
])
gene_df = gene_df.sort_values(['family', 'gene_symbol'])
print(f"Total genes: {len(gene_df)}")
gene_df

## 2. Microarray dataset overview

In [ ]:
# Load sample annotations for one donor
samples = data_loader.load_microarray_samples(9861)
print(f"Donor 9861: {len(samples)} samples")
print(f"Unique structures: {samples['structure_acronym'].nunique()}")
print(f"\nColumns: {samples.columns.tolist()}")
samples.head()

In [ ]:
# Distribution of samples across main brain structures
fig, ax = plt.subplots(figsize=(12, 5))
structure_counts = samples['structure_name'].value_counts().head(30)
structure_counts.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Number of Samples')
ax.set_title('Top 30 Brain Structures by Sample Count (Donor 9861)')
ax.invert_yaxis()
plt.tight_layout()

## 3. RNA-seq dataset overview

In [ ]:
rnaseq_samples = data_loader.load_rnaseq_samples(9861)
print(f"RNA-seq donor 9861: {len(rnaseq_samples)} samples")
print(f"Unique structures: {rnaseq_samples['ontology_structure_acronym'].nunique()}")
rnaseq_samples.head()

## 4. Mechanosensitive channel expression — single donor

In [ ]:
genes = list(gene_lists.ALL_MECHANOSENSITIVE.keys())
expr = data_loader.get_microarray_gene_expression(genes, donor_id=9861)
print(f"Expression matrix: {expr.shape[0]} genes x {expr.shape[1]} samples")
expr.iloc[:5, :5]

In [ ]:
# Overall expression levels across all samples
mean_expr = expr.mean(axis=1).sort_values(ascending=False)
display_names = [gene_lists.get_display_name(g) for g in mean_expr.index]
families = [gene_lists.get_family(g) for g in mean_expr.index]

palette = {
    'Piezo': '#E74C3C', 'K2P (mechano)': '#3498DB', 'K2P (other)': '#85C1E9',
    'TRPV': '#E67E22', 'TRPA': '#F39C12', 'TRPC': '#27AE60',
    'TRPM': '#1ABC9C', 'TRPP': '#8E44AD',
}
colors = [palette.get(f, '#95A5A6') for f in families]

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(display_names, mean_expr.values, color=colors)
ax.set_xlabel('Mean Expression (Microarray, Donor 9861)', fontsize=12)
ax.set_title('Mechanosensitive Ion Channel Expression Levels', fontsize=14)
ax.invert_yaxis()

# Add legend
from matplotlib.patches import Patch
used_families = sorted(set(families))
legend_patches = [Patch(facecolor=palette[f], label=f) for f in used_families]
ax.legend(handles=legend_patches, title='Channel Family', loc='lower right', fontsize=9)
plt.tight_layout()

## 5. RNA-seq expression comparison

In [ ]:
rnaseq_expr = data_loader.get_rnaseq_gene_expression(genes, donor_id=9861)
print(f"RNA-seq: {rnaseq_expr.shape[0]} genes x {rnaseq_expr.shape[1]} samples")

# Mean expression (TPM)
mean_tpm = rnaseq_expr.mean(axis=1).sort_values(ascending=False)
display_names_rna = [gene_lists.get_display_name(g) for g in mean_tpm.index]
families_rna = [gene_lists.get_family(g) for g in mean_tpm.index]
colors_rna = [palette.get(f, '#95A5A6') for f in families_rna]

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(display_names_rna, mean_tpm.values, color=colors_rna)
ax.set_xlabel('Mean TPM (RNA-seq, Donor 9861)', fontsize=12)
ax.set_title('Mechanosensitive Ion Channel Expression — RNA-seq', fontsize=14)
ax.set_xscale('log')
ax.invert_yaxis()
ax.legend(handles=legend_patches, title='Channel Family', loc='lower right', fontsize=9)
plt.tight_layout()

## 6. Expression variability across brain regions

Which genes show the most variation across brain regions? High variability suggests region-specific roles.

In [ ]:
cv = (expr.std(axis=1) / expr.mean(axis=1)).sort_values(ascending=False)
cv_names = [gene_lists.get_display_name(g) for g in cv.index]

fig, ax = plt.subplots(figsize=(12, 8))
cv_families = [gene_lists.get_family(g) for g in cv.index]
cv_colors = [palette.get(f, '#95A5A6') for f in cv_families]
ax.barh(cv_names, cv.values, color=cv_colors)
ax.set_xlabel('Coefficient of Variation', fontsize=12)
ax.set_title('Expression Variability Across Brain Regions', fontsize=14)
ax.invert_yaxis()
ax.legend(handles=legend_patches, title='Channel Family', loc='lower right', fontsize=9)
plt.tight_layout()